# 01 — Acquisition & préparation (données réelles)
## Démographie, santé & conditions de vie au Sénégal — EDS/DHS & Banque mondiale

**100 % données réelles, via APIs publiques** (téléchargées par
`scripts/download_data.py`) :
- 👶 **DHS / EDS** (api.dhsprogram.com) — enquêtes démographiques et de santé
  menées avec l'ANSD : **séries nationales 1986→2023** + **détail régional (EDS 2023)**.
- 🌍 **Banque mondiale** — pauvreté, Gini, population, structure par âge, urbanisation.
- 🗺️ **geoBoundaries** — 14 régions.

> Les microdonnées EHCVM/RGPH nécessitent une inscription (ANADS) ; on exploite ici
> les **indicateurs officiels** issus de ces enquêtes, exposés par ces APIs.


In [1]:

import os, warnings, json, re, unicodedata
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110, "axes.titlesize": 13})

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw"); PROC = os.path.join(PROJ, "data", "processed")
GEO = os.path.join(PROJ, "data", "geo"); FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS): os.makedirs(d, exist_ok=True)

def deacc(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")
print("Racine projet :", PROJ)


Racine projet : C:\projet\senegal-demographie


### 1. Chargement

In [2]:

dhs_nat = pd.read_csv(os.path.join(RAW, "dhs_national.csv"))
dhs_sub = pd.read_csv(os.path.join(RAW, "dhs_subnational_2023.csv"))
wb = pd.read_csv(os.path.join(RAW, "worldbank.csv"))
print("DHS national :", dhs_nat.shape, "| DHS régional :", dhs_sub.shape, "| WB :", wb.shape)
dhs_nat.head(3)


DHS national : (202, 4) | DHS régional : (231, 4) | WB : (535, 4)


,annee,code,indicateur,valeur
0,1986,CM_ECMR_C_IMR,Mortalité infantile (‰),88.0
1,1986,CM_ECMR_C_IMR,Mortalité infantile (‰),91.0
2,1993,CM_ECMR_C_IMR,Mortalité infantile (‰),68.0


### 2. Nettoyage du détail régional
Les libellés DHS sont hiérarchiques (zones-agrégats sans préfixe ; régions avec
`..` ; régions créées en 2008 avec `....` ; doublons d'anciennes limites `(2005)`).
On isole les **14 régions administratives actuelles**.

In [3]:

EXCL = {"Nord et Est", "Ouest", "Centre", "Sud"}     # zones-agrégats à exclure
def clean_region(lbl):
    s = re.sub(r"^[.\s]+", "", str(lbl))             # enlève les points de hiérarchie
    s = re.sub(r"\s*\(\d{4}\)\s*", "", s).strip()    # enlève (2005)/(2010)
    return s
sub = dhs_sub.copy()
sub = sub[~sub["region_label"].str.contains(r"\(2005\)")]   # écarte anciennes limites
sub["region"] = sub["region_label"].map(clean_region)
sub = sub[~sub["region"].isin(EXCL)]                        # écarte les zones-agrégats
sub["region_geo"] = sub["region"].map(deacc)               # clé de jointure GeoJSON (ASCII)
print("Régions retenues (", sub["region"].nunique(), "):", sorted(sub["region"].unique()))


Régions retenues ( 14 ): ['Dakar', 'Diourbel', 'Fatick', 'Kaffrine', 'Kaolack', 'Kolda', 'Kédougou', 'Louga', 'Matam', 'Saint Louis', 'Sédhiou', 'Tambacounda', 'Thiès', 'Ziguinchor']


### 3. Tables larges + dimension des indicateurs

In [4]:

dim_indicator = (dhs_nat[["code", "indicateur"]].drop_duplicates()
                 .assign(source="DHS/EDS"))
dhs_nat_wide = dhs_nat.pivot_table(index="annee", columns="code", values="valeur").reset_index()
dhs_sub_wide = sub.pivot_table(index=["region", "region_geo"], columns="code", values="valeur").reset_index()
wb_wide = wb.pivot_table(index="annee", columns="code", values="valeur").reset_index()
print("national large :", dhs_nat_wide.shape, "| régional large :", dhs_sub_wide.shape)
dhs_sub_wide.head(3)


national large : (16, 11) | régional large : (14, 12)


code,region,region_geo,CM_ECMR_C_IMR,CM_ECMR_C_NNR,CM_ECMR_C_U5M,CN_NUTS_C_HA2,ED_LITR_W_LIT,FE_FRTR_W_TFR,FP_CUSM_W_MOD,HC_ELEC_H_ELC,RH_DELP_C_DHF,WS_SRCE_H_IMP
0,Dakar,Dakar,20.0,16.0,25.0,10.4,68.9,3.1,36.4,98.2,98.70,99.9
1,Diourbel,Diourbel,39.0,30.0,52.0,17.5,30.7,4.1,16.6,87.0,94.05,99.4
2,Fatick,Fatick,25.0,22.0,36.0,15.9,52.9,4.6,27.1,55.7,91.95,82.4


### 4. Validation — valeurs nationales EDS 2023 & amplitude régionale

In [5]:

last = dhs_nat[dhs_nat["annee"] == dhs_nat["annee"].max()]
print("== Indicateurs nationaux EDS", int(dhs_nat['annee'].max()), "==")
for _, r in last.iterrows():
    print(f"  {r['valeur']:>6}  {r['indicateur']}")
tfr = sub[sub["code"] == "FE_FRTR_W_TFR"]
print("\nFécondité régionale 2023 : de", tfr['valeur'].min(), "(",
      tfr.loc[tfr['valeur'].idxmin(),'region'], ") à", tfr['valeur'].max(),
      "(", tfr.loc[tfr['valeur'].idxmax(),'region'], ")")


== Indicateurs nationaux EDS 2023 ==
    31.0  Mortalité infantile (‰)
    33.0  Mortalité infantile (‰)
    23.0  Mortalité néonatale (‰)
    24.0  Mortalité néonatale (‰)
    40.0  Mortalité des moins de 5 ans (‰)
    42.0  Mortalité des moins de 5 ans (‰)
    17.5  Retard de croissance / malnutrition chronique (%)
    50.6  Alphabétisation des femmes (%)
     4.0  Indice de fécondité (enfants/femme)
    25.6  Contraception moderne, femmes mariées (%)
    77.0  Ménages avec électricité (%)
    92.1  Accouchements en établissement de santé (%)
    92.3  Accouchements en établissement de santé (%)
    90.9  Accès à une source d'eau améliorée (%)

Fécondité régionale 2023 : de 3.1 ( Dakar ) à 6.0 ( Kaffrine )


### 5. Écriture dans `data/processed/`

In [6]:

tables = {"dhs_national": dhs_nat, "dhs_national_wide": dhs_nat_wide,
          "dhs_subnational": sub[["region","region_geo","code","indicateur","valeur"]],
          "dhs_subnational_wide": dhs_sub_wide,
          "worldbank": wb, "worldbank_wide": wb_wide, "dim_indicator": dim_indicator}
for name, df in tables.items():
    df.to_csv(os.path.join(PROC, f"{name}.csv"), index=False, encoding="utf-8-sig")
    print(f"  {name:22s} {df.shape}")
print("\n✅ Données réelles préparées.")


  dhs_national           (202, 4)
  dhs_national_wide      (16, 11)
  dhs_subnational        (154, 5)
  dhs_subnational_wide   (14, 12)
  worldbank              (535, 4)
  worldbank_wide         (65, 12)
  dim_indicator          (10, 3)

✅ Données réelles préparées.
